# Content-Profile Coverage Check

Run before building Tier 4 (hybrid/content-aware): does a *user's* odds of having any content signal at all depend on how many books they've read, even though *book-level* description coverage is flat across popularity (checked separately in `profile_data.ipynb`)?

The report-building logic lives in `src/bxbench/profile.py` (`compute_content_coverage_by_bucket` / `render_content_coverage_markdown`) so this notebook is for interactive display; the last cell writes the canonical `reports/content_coverage.md`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from bxbench.data import load_descriptions
from bxbench.profile import compute_content_coverage_by_bucket, render_content_coverage_markdown

train = pd.read_parquet(Path.cwd().parent / "data" / "processed" / "train.parquet")
descriptions = load_descriptions()

coverage = compute_content_coverage_by_bucket(train, descriptions)
print(f"train items: {coverage['n_train_items']:,}")
print(f"described items: {coverage['n_desc_items']:,}")
print(f"overlap (described AND in train): {coverage['n_overlap']:,}")
coverage["by_bucket"]

train items: 318,264
described items: 7,021
overlap (described AND in train): 6,842


,bucket,sum,count,pct
0,1-2,457,19036,2.400714
1,3-5,662,9698,6.826150
2,6-20,1951,10562,18.471880
3,21+,4395,6821,64.433368


Per-user content-profile coverage rises sharply with history depth (2.4% at 1-2 interactions vs. 64.4% at 21+) even though per-book description coverage is flat. A user needs at least one described book among their train interactions to get a content profile at all -- more interactions means more rolls of the same ~2.6%-per-book dice, not a higher per-book chance.

**Implication for Tier 4:** whatever content signal exists will be concentrated almost entirely in the bucket that already has the most collaborative signal to work with (deep history), and nearly absent from the bucket where it would matter most (sparse/cold-start users) -- worth expecting a weak measured lift going in, not a surprise coming out.

In [2]:
report_path = Path.cwd().parent / "reports" / "content_coverage.md"
report_path.parent.mkdir(exist_ok=True)
report_path.write_text(render_content_coverage_markdown(coverage))
print(f"Wrote {report_path}")

Wrote /Users/patrickcher/Library/CloudStorage/GoogleDrive-patrickcher@gmail.com/My Drive/AI Thoughts/2026-09-book-crossing-personalization-benchmark/reports/content_coverage.md
